# [Utilities functions - RDFLib](https://rdflib.readthedocs.io/en/stable/utilities/)

## Functional properties

functional property: つまりリソースごとに 1 回しか出現しないプロパティ

ただし，オントロジー的制約を追加するというより， `set` の挙動が整合するってだけ

In [ ]:
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import FOAF, RDF

g: Graph = Graph()
g.bind("foaf", FOAF)

# Add demo data
bob: URIRef = URIRef("http://example.org/people/Bob")

g.add((bob, RDF.type, FOAF.Person))
g.add((bob, FOAF.name, Literal("Bob")))
g.add((bob, FOAF.age, Literal(38)))

# To get a single value, use 'value'
print(g.value(bob, FOAF.age))

# To change a single value, use 'set'
g.set((bob, FOAF.age, Literal(39)))
print(g.value(bob, FOAF.age))

38
39


## Slicing graphs

通常 Pythonのリストに対するスライス挙動と違うことに注意！

| コード | マッチング条件 | 戻り値 |
| :--- | :--- | :--- |
| `g[:]` | (?, ?, ?) | (s, p, o) |
| `g[bob]` | (bob, ?, ?) | (p, o) |
| `g[:FOAF.knows]` | (?, knows, ?) | (s, o) |
| `g[::bill]` | (?, ?, bill) | (s, p) |
| `g[bob:FOAF.knows]` | (bob, knows, ?) | o |
| `g[bob::bill]` | (bob, ?, bill) | p |
| `g[:FOAF.knows:bill]` | (?, knows, bill) | s |
| `g[bob:FOAF.knows:bill]` | (bob, knows, bill) | bool |

In [3]:
for i in range(20)[2:9:3]:
    print(i)


2
5
8


In [4]:
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import FOAF, RDF

g: Graph = Graph()
g.bind("foaf", FOAF)

# Add demo data
bob: URIRef = URIRef("http://example.org/people/Bob")
bill: URIRef = URIRef("http://example.org/people/Bill")

g.add((bob, RDF.type, FOAF.Person))
g.add((bob, FOAF.name, Literal("Bob")))
g.add((bob, FOAF.age, Literal(38)))
g.add((bob, FOAF.knows, bill))

<Graph identifier=Nc30da962f59a4517b22f4b55ffb2bb02 (<class 'rdflib.graph.Graph'>)>

In [5]:
print(list(g[:]))

# same as
print(list(iter(g)))

[(rdflib.term.URIRef('http://example.org/people/Bob'), rdflib.term.URIRef('http://xmlns.com/foaf/0.1/name'), rdflib.term.Literal('Bob')), (rdflib.term.URIRef('http://example.org/people/Bob'), rdflib.term.URIRef('http://xmlns.com/foaf/0.1/knows'), rdflib.term.URIRef('http://example.org/people/Bill')), (rdflib.term.URIRef('http://example.org/people/Bob'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://xmlns.com/foaf/0.1/Person')), (rdflib.term.URIRef('http://example.org/people/Bob'), rdflib.term.URIRef('http://xmlns.com/foaf/0.1/age'), rdflib.term.Literal('38', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))]
[(rdflib.term.URIRef('http://example.org/people/Bob'), rdflib.term.URIRef('http://xmlns.com/foaf/0.1/name'), rdflib.term.Literal('Bob')), (rdflib.term.URIRef('http://example.org/people/Bob'), rdflib.term.URIRef('http://xmlns.com/foaf/0.1/knows'), rdflib.term.URIRef('http://example.org/people/Bill')), (rdfli

In [6]:
print(list(g[bob : FOAF.knows]))

# same as
print(list(g.objects(bob, FOAF.knows)))

[rdflib.term.URIRef('http://example.org/people/Bill')]
[rdflib.term.URIRef('http://example.org/people/Bill')]


In [7]:
print(g[bob : FOAF.knows : bill])

# same as
print((bob, FOAF.knows, bill) in g)

True
True


In [8]:
print(list(g[: FOAF.knows]))

# same as
print(list(g.subject_objects(FOAF.knows)))

[(rdflib.term.URIRef('http://example.org/people/Bob'), rdflib.term.URIRef('http://example.org/people/Bill'))]
[(rdflib.term.URIRef('http://example.org/people/Bob'), rdflib.term.URIRef('http://example.org/people/Bill'))]


## Serializing a single term to N3

In [16]:
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import FOAF

# A URIRef
person: URIRef = URIRef("http://xmlns.com/foaf/0.1/Person")

print(person)
print(person.n3())

http://xmlns.com/foaf/0.1/Person
<http://xmlns.com/foaf/0.1/Person>


In [12]:
# Simplifying the output with a namespace prefix
g: Graph = Graph()
g.bind("foaf", FOAF)

print(person.n3(g.namespace_manager))

foaf:Person


In [13]:
# A typed literal
l: Literal = Literal(2)

print(l.n3())

"2"^^<http://www.w3.org/2001/XMLSchema#integer>


In [14]:
# Simplifying the output with a namespace prefix
# XSD is built in, so no need to bind() it!
print(l.n3(g.namespace_manager))

"2"^^xsd:integer


## SPARQL Paths

In [20]:
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import FOAF
from rdflib.paths import OneOrMore, ZeroOrMore, ZeroOrOne

g: Graph = Graph()
ex: Namespace = Namespace("https://example.org/")
g.bind("ex", ex)
g.bind("foaf", FOAF)

connections: tuple[tuple[URIRef, URIRef], ...] = (
    (ex.alice, ex.bob),
    (ex.bob, ex.carol),
    (ex.carol, ex.dave),
)
for src, dst in connections:
    g.add((src, FOAF.knows, dst))

names: tuple[tuple[URIRef, str], ...] = (
    (ex.alice, "Alice"),
    (ex.bob, "Bob"),
    (ex.carol, "Carol"),
    (ex.dave, "Dave"),
)
for person, name in names:
    g.add((person, FOAF.name, Literal(name)))

g.add((ex.bob, FOAF.nick, Literal("Bobby")))

print(f"triples: {len(g)}")


triples: 8


In [22]:
# 連結

print("Two knows hops from Alice:")
print(
    sorted(
        term.n3(g.namespace_manager)
        for term in g.objects(
            ex.alice,
            FOAF.knows / FOAF.knows,
        )
    )
)

print("Names of Alice's direct acquaintances:")
print(
    sorted(
        term.n3(g.namespace_manager)
        for term in g.objects(
            ex.alice,
            FOAF.knows / FOAF.name,
        )
    )
)

Two knows hops from Alice:
['ex:carol']
Names of Alice's direct acquaintances:
['"Bob"']


In [23]:
# 逆向き

print("Who knows Bob?")
print(
    sorted(
        term.n3(g.namespace_manager)
        for term in g.objects(
            ex.bob,
            ~FOAF.knows,
        )
    )
)

print("Equivalent single-edge lookup:")
print(sorted(term.n3(g.namespace_manager) for term in g.subjects(FOAF.knows, ex.bob)))

Who knows Bob?
['ex:alice']
Equivalent single-edge lookup:
['ex:alice']


In [24]:
# 選択 (|)

print("Bob's name OR nickname:")
print(
    sorted(
        term.n3(g.namespace_manager)
        for term in g.objects(
            ex.bob,
            FOAF.name | FOAF.nick,
        )
    )
)

Bob's name OR nickname:
['"Bob"', '"Bobby"']


In [26]:
# 反復

print("OneOrMore (+):")
print(
    sorted(
        term.n3(g.namespace_manager)
        for term in g.objects(
            ex.alice,
            FOAF.knows * OneOrMore,
        )
    )
)

print("ZeroOrMore (*):")
print(
    sorted(
        term.n3(g.namespace_manager)
        for term in g.objects(
            ex.alice,
            FOAF.knows * ZeroOrMore,
        )
    )
)

print("ZeroOrOne (?):")
print(
    sorted(
        term.n3(g.namespace_manager)
        for term in g.objects(
            ex.alice,
            FOAF.knows * ZeroOrOne,
        )
    )
)

OneOrMore (+):
['ex:bob', 'ex:carol', 'ex:dave']
ZeroOrMore (*):
['ex:alice', 'ex:bob', 'ex:carol', 'ex:dave']
ZeroOrOne (?):
['ex:alice', 'ex:bob']


In [27]:
# SPARQL でも
sparql: str = """
PREFIX ex: <https://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name WHERE {
    ex:alice foaf:knows+ / foaf:name ?name .
}
ORDER BY ?name
"""

print("Names reachable via one or more knows edges:")
for row in g.query(sparql):
    print(str(row.name))

Names reachable via one or more knows edges:
Bob
Carol
Dave
